# Tomahawk PF MGRID — shared-grid native worker Part 1

Run Parts 1, 2, and 3 simultaneously in separate Kaggle sessions. Attach the PF handoff ZIP and the existing `x-grid-zip` runtime dataset to each session, then select **Run All**. The notebook also attempts KaggleHub retrieval of `nicolebryant29/x-grid-zip` when Internet is enabled.

This is the PF branch: six PF groups, `nfp=1`, and the supplied PF currents. It does not relabel PF as 48 TF coils or claim that PF alone produces 16 T. The common MGRID numerical grid is `R=4…11 m`, `Z=-3.5…+3.5 m`, `KP=36`, `IR=JZ=201`, mode `S`, no stellarator symmetry. The companion TF branch remains the verified 48-physical-coil / 16-field-period / 16 T branch with the strict 0.7539% reference ripple below the 1% limit.

Only a valid native PF partial MGRID produces `PF_MGRID_PART_PASS`. Missing XGRID, failed execution, invalid output, or manifest mismatch raises a visible error after a diagnostic ZIP is written. There is no internal XGRID timeout by default; heartbeats print every 60 seconds.


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import hashlib
import json
import math
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

try:
    import numpy as np
    from scipy.io import netcdf_file
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'numpy', 'scipy'])
    import numpy as np
    from scipy.io import netcdf_file

PART_INDEX = 1
COIL_FILE = f'coils.tomahawk_pf_part{PART_INDEX}'
INPUT_FILE = f'input_xgrid_tomahawk_pf_part{PART_INDEX}'
EXTCUR_FILE = f'extcur.tomahawk_pf_part{PART_INDEX}'
EXPECTED_MGRID = f'mgrid_tomahawk_pf_part{PART_INDEX}.nc'
PF_GROUPS = ['PF_L_IN', 'PF_L_OUT']
PF_CURRENTS = np.asarray([19123.4976703, 19904.2740987], dtype=float)
PF_LOOPS = 190
FULL_PF_GROUPS = ['PF_L_IN', 'PF_L_OUT', 'PF_M_IN', 'PF_M_OUT', 'PF_U_IN', 'PF_U_OUT']
SHARED_GRID = {'rmin_m': 4.0, 'rmax_m': 11.0, 'zmin_m': -3.5, 'zmax_m': 3.5,
               'toroidal_planes': 36, 'radial_points': 201, 'vertical_points': 201,
               'mgrid_mode': 'S', 'stellarator_symmetry': False}
FIELD_VARS = tuple(name for i in range(1, len(PF_GROUPS) + 1) for name in (f'br_{i:03d}', f'bp_{i:03d}', f'bz_{i:03d}'))


class WorkerFailure(RuntimeError):
    pass


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def env_seconds(name: str, default: float) -> float:
    raw = os.environ.get(name, str(default)).strip()
    value = float(raw)
    if not math.isfinite(value) or value < 0:
        raise ValueError(f'{name} must be finite and non-negative')
    return value


def safe_extract(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            target = (destination / info.filename).resolve()
            if target != base and base not in target.parents:
                raise RuntimeError(f'unsafe archive member: {info.filename}')
        zf.extractall(destination)


def search_roots(work: Path) -> list[Path]:
    values = []
    if os.environ.get('PF_MGRID_DATA_ROOT'):
        values.append(Path(os.environ['PF_MGRID_DATA_ROOT']).expanduser())
    here = Path.cwd().resolve()
    values.extend([here, here.parent, here.parent.parent, work, work.parent])
    values.extend(Path(item) for item in ('/kaggle/input', '/kaggle/working', '/kaggle/models', '/mnt/data'))
    result, seen = [], set()
    for value in values:
        try:
            value = value.resolve()
        except OSError:
            continue
        if value == Path('/') or not value.exists() or value in seen:
            continue
        result.append(value)
        seen.add(value)
    return result


def package_archives(roots: list[Path]) -> list[Path]:
    found, seen = [], set()
    for root in roots:
        if root.is_file() and root.suffix.lower() == '.zip':
            items = [root]
        elif root.is_dir():
            try:
                items = list(root.rglob('*.zip'))
            except OSError:
                items = []
        else:
            items = []
        for item in items:
            if item not in seen and any(token in item.name.lower() for token in ('pf', 'tomahawk', 'x-grid', 'xgrid', 'makegrid', 'stellopt')):
                found.append(item)
                seen.add(item)
    return found


def parse_coil_summary(path: Path) -> dict:
    counts = {name: 0 for name in FULL_PF_GROUPS}
    with path.open() as f:
        for line in f:
            fields = line.split()
            if len(fields) >= 6 and fields[-1] in counts:
                counts[fields[-1]] += 1
    selected = {name: counts[name] for name in PF_GROUPS}
    return {'loops_by_group': selected, 'loops': sum(selected.values()), 'groups': [name for name in PF_GROUPS if counts[name]]}


def validate_root(root: Path) -> tuple[bool, list[str]]:
    errors = []
    contract_path = root / 'pf_mgrid_input_contract.json'
    manifest_path = root / 'dataset_manifest.json'
    if not contract_path.is_file() or not manifest_path.is_file():
        return False, ['PF contract or dataset_manifest missing']
    try:
        contract = json.loads(contract_path.read_text())
        manifest = json.loads(manifest_path.read_text())
    except Exception as exc:
        return False, [f'invalid JSON: {exc}']
    shared = contract.get('shared_grid', {})
    for key, expected in SHARED_GRID.items():
        if shared.get(key) != expected:
            errors.append(f'shared_grid {key}={shared.get(key)!r}, expected {expected!r}')
    branch = contract.get('pf_branch', {})
    if branch.get('nfp') != 1:
        errors.append('PF nfp must be 1')
    if branch.get('full_group_count') != 6:
        errors.append('PF full group count must be 6')
    chunk = contract.get('chunk', {})
    part_definition = chunk.get('part_definitions', {}).get(str(PART_INDEX), {})
    if part_definition.get('groups') != PF_GROUPS:
        errors.append(f'contract groups {part_definition.get("groups")} != {PF_GROUPS}')
    if part_definition.get('loops') != PF_LOOPS:
        errors.append(f'contract loops {part_definition.get("loops")} != {PF_LOOPS}')
    for entry in manifest.get('files', []):
        path = root / entry['path']
        if not path.is_file():
            errors.append(f'missing {entry["path"]}')
        elif sha256(path) != entry.get('sha256'):
            errors.append(f'SHA-256 mismatch {entry["path"]}')
    if manifest.get('contract_sha256') != sha256(contract_path):
        errors.append('contract SHA-256 mismatch')
    return not errors, errors


def find_data_root(work: Path) -> tuple[Path | None, list[str]]:
    roots = search_roots(work)
    candidates = []
    for root in roots:
        if root.is_dir():
            try:
                candidates.extend(path.parent for path in root.rglob('pf_mgrid_input_contract.json'))
            except OSError:
                pass
    extracted = work / 'input_archives'
    for archive in package_archives(roots):
        destination = extracted / archive.stem
        marker = destination / '.extracted'
        if not marker.exists():
            try:
                safe_extract(archive, destination)
                marker.write_text('ok\n')
            except Exception:
                continue
        try:
            candidates.extend(path.parent for path in destination.rglob('pf_mgrid_input_contract.json'))
        except OSError:
            pass
    errors, seen = [], set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        valid, candidate_errors = validate_root(candidate)
        if valid:
            return candidate, []
        errors.extend([f'{candidate}: {error}' for error in candidate_errors])
    return None, errors


def find_xgrid(roots: list[Path], work: Path) -> str | None:
    def scan(search: list[Path]) -> str | None:
        candidates = []
        if os.environ.get('STELLOPT_XGRID'):
            candidates.append(os.environ['STELLOPT_XGRID'])
        candidates.extend(['/home/STELLOPT/MAKEGRID/Release/xgrid', '/usr/local/bin/xgrid', shutil.which('xgrid') or ''])
        kaggle_roots = [root for root in search if str(root).startswith('/kaggle/input')]
        other_roots = [root for root in search if root not in kaggle_roots]
        for root in kaggle_roots + other_roots:
            if root.is_dir():
                try:
                    candidates.extend(str(path) for path in root.rglob('xgrid') if path.is_file())
                except OSError:
                    pass
        for item in candidates:
            try:
                parts = shlex.split(item)
            except ValueError:
                continue
            if len(parts) == 1 and Path(parts[0]).is_file():
                return parts[0]
            if len(parts) == 1 and shutil.which(parts[0]):
                return shutil.which(parts[0])
        return None

    found = scan(roots)
    if found:
        return found
    if os.environ.get('PF_MGRID_DISABLE_KAGGLEHUB', '').strip().lower() in {'1', 'true', 'yes'}:
        print(f'PF_MGRID_PART{PART_INDEX}_KAGGLEHUB_DISABLED', flush=True)
        return None
    handle = os.environ.get('PF_MGRID_XGRID_DATASET_HANDLE', 'nicolebryant29/x-grid-zip').strip()
    if not handle:
        return None
    print(f'PF_MGRID_PART{PART_INDEX}_XGRID_DATASET_DOWNLOAD_ATTEMPT={handle}', flush=True)
    try:
        try:
            import kagglehub
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'kagglehub'])
            import kagglehub
        downloaded = Path(kagglehub.dataset_download(handle)).expanduser().resolve()
        downloaded_roots = [downloaded]
        for archive in package_archives(downloaded_roots):
            destination = work / 'input_archives' / f'kagglehub_{archive.stem}'
            marker = destination / '.extracted'
            if not marker.exists():
                safe_extract(archive, destination)
                marker.parent.mkdir(parents=True, exist_ok=True)
                marker.write_text('ok\n')
            downloaded_roots.append(destination)
        found = scan(downloaded_roots)
        if found:
            print(f'PF_MGRID_PART{PART_INDEX}_XGRID_DATASET_FOUND={found}', flush=True)
            return found
        print(f'PF_MGRID_PART{PART_INDEX}_XGRID_DATASET_NO_EXECUTABLE={downloaded}', flush=True)
    except Exception as exc:
        print(f'PF_MGRID_PART{PART_INDEX}_XGRID_DATASET_DOWNLOAD_FAILED={exc!r}', flush=True)
    return None


def stage_xgrid(source: str, native_dir: Path) -> tuple[Path, dict]:
    source_path = Path(source)
    staged = native_dir / 'xgrid'
    shutil.copy2(source_path, staged)
    staged.chmod(staged.stat().st_mode | 0o111)
    environment = os.environ.copy()
    library_paths = []
    for candidate in (source_path.parent / 'lib', source_path.parent / 'lib64'):
        if candidate.is_dir():
            target = native_dir / candidate.name
            shutil.copytree(candidate, target, dirs_exist_ok=True)
            library_paths.append(str(target))
    if library_paths:
        environment['LD_LIBRARY_PATH'] = os.pathsep.join(library_paths + ([environment['LD_LIBRARY_PATH']] if environment.get('LD_LIBRARY_PATH') else []))
    return staged, environment


def run_native(executable: Path, input_path: Path, native_dir: Path, log_path: Path, environment: dict) -> tuple[int, float, bool]:
    heartbeat = max(0.1, env_seconds('PF_MGRID_HEARTBEAT_SECONDS', 60.0))
    timeout = env_seconds('PF_MGRID_XGRID_TIMEOUT_SECONDS', 0.0)
    started = time.time()
    print(f'PF_MGRID_PART{PART_INDEX}_NATIVE_XGRID_START timeout={"UNLIMITED" if timeout == 0 else f"{timeout:.0f}s"}', flush=True)
    with input_path.open() as stdin_file, log_path.open('w') as log_file:
        process = subprocess.Popen([str(executable)], stdin=stdin_file, cwd=native_dir, text=True, stdout=log_file, stderr=subprocess.STDOUT, env=environment)
        while True:
            try:
                code = process.wait(timeout=heartbeat)
                elapsed = time.time() - started
                print(f'PF_MGRID_PART{PART_INDEX}_NATIVE_XGRID_END return_code={code} elapsed_s={elapsed:.1f}', flush=True)
                return code, elapsed, False
            except subprocess.TimeoutExpired:
                elapsed = time.time() - started
                print(f'PF_MGRID_PART{PART_INDEX}_NATIVE_XGRID_RUNNING elapsed_s={elapsed:.1f} pid={process.pid}', flush=True)
                if timeout > 0 and elapsed >= timeout:
                    process.terminate()
                    try:
                        code = process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        code = process.wait()
                    return code, time.time() - started, True


def decode_groups(data: np.ndarray) -> list[str]:
    if data.ndim != 2:
        return []
    result = []
    for row in data:
        result.append(b''.join(np.asarray(row).astype('S1').tolist()).decode('ascii', errors='ignore').rstrip())
    return result


def inspect_mgrid(path: Path) -> dict:
    errors = []
    try:
        with netcdf_file(str(path), 'r', mmap=False) as nc:
            dimensions = {key: int(value) for key, value in nc.dimensions.items()}
            variables = {key: np.array(value.data).copy() for key, value in nc.variables.items()}
        expected_dims = {'phi': 36, 'zee': 201, 'rad': 201, 'external_coil_groups': len(PF_GROUPS), 'external_coils': len(PF_GROUPS)}
        for name, expected in expected_dims.items():
            if dimensions.get(name) != expected:
                errors.append(f'dimension {name}={dimensions.get(name)!r}, expected {expected}')
        for name in FIELD_VARS:
            if name not in variables or variables[name].shape != (36, 201, 201):
                errors.append(f'missing or malformed {name}')
            elif not np.all(np.isfinite(variables[name])):
                errors.append(f'non-finite values in {name}')
            elif not np.any(variables[name]):
                errors.append(f'zero field array {name}')
        mode = ''.join(np.asarray(variables.get('mgrid_mode', [])).astype('U1').reshape(-1).tolist())
        if mode != 'S':
            errors.append(f'mgrid_mode={mode!r}, expected S')
        nfp = int(round(float(np.asarray(variables.get('nfp', [0])).reshape(-1)[0]))) if 'nfp' in variables else None
        if nfp != 1:
            errors.append(f'nfp={nfp!r}, expected 1')
        raw = np.asarray(variables.get('raw_coil_cur', []), dtype=float).reshape(-1)
        if raw.size != len(PF_GROUPS) or not np.allclose(raw, PF_CURRENTS, rtol=0, atol=1e-3):
            errors.append(f'raw_coil_cur={raw.tolist()}, expected {PF_CURRENTS.tolist()}')
        groups = decode_groups(variables.get('coil_group', np.empty((0, 0))))
        if groups != PF_GROUPS:
            errors.append(f'coil_group={groups!r}, expected {PF_GROUPS!r}')
        scalar_expectations = {'ir': 201, 'jz': 201, 'kp': 36}
        for name, expected in scalar_expectations.items():
            if name not in variables or int(round(float(np.asarray(variables[name]).reshape(-1)[0]))) != expected:
                errors.append(f'{name} metadata mismatch')
        for name, expected in {'rmin': 4.0, 'rmax': 11.0, 'zmin': -3.5, 'zmax': 3.5}.items():
            if name not in variables or not np.isclose(float(np.asarray(variables[name]).reshape(-1)[0]), expected, atol=1e-10):
                errors.append(f'{name} metadata mismatch')
        return {'path': str(path), 'sha256': sha256(path), 'dimensions': dimensions, 'mode': mode, 'nfp': nfp, 'coil_groups': groups, 'raw_coil_cur_A': raw.tolist(), 'errors': errors, 'valid': not errors}
    except Exception as exc:
        return {'path': str(path), 'errors': [f'MGRID read failed: {exc}'], 'valid': False}


def make_output_zip(work: Path, data_root: Path, report_path: Path, native_dir: Path | None) -> Path:
    output = work / f'PF_MGRID_PART{PART_INDEX}_OUTPUT.zip'
    with zipfile.ZipFile(output, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(report_path, report_path.name)
        for name in (COIL_FILE, INPUT_FILE, EXTCUR_FILE):
            path = data_root / name
            if path.is_file():
                zf.write(path, name)
        if native_dir and native_dir.is_dir():
            for path in native_dir.iterdir():
                if path.is_file():
                    zf.write(path, f'native/{path.name}')
    return output


def print_log_tail(path: Path, lines: int = 50) -> None:
    if path.is_file():
        print(f'PF_MGRID_PART{PART_INDEX}_XGRID_LOG_TAIL_START', flush=True)
        for line in path.read_text(errors='replace').splitlines()[-lines:]:
            print(line, flush=True)
        print(f'PF_MGRID_PART{PART_INDEX}_XGRID_LOG_TAIL_END', flush=True)


def main() -> None:
    started = time.time()
    if os.environ.get('PF_MGRID_WORKDIR'):
        work = Path(os.environ['PF_MGRID_WORKDIR']).expanduser()
    elif Path('/kaggle/working').is_dir():
        work = Path('/kaggle/working') / f'pf_mgrid_part{PART_INDEX}_run'
    else:
        work = Path.cwd() / f'pf_mgrid_part{PART_INDEX}_run'
    work.mkdir(parents=True, exist_ok=True)
    report_path = work / f'PF_MGRID_PART{PART_INDEX}_REPORT.json'
    report = {
        'schema_version': 'TOMAHAWK-PF-SHARED-GRID-PART-REPORT-1.0-HARD-FAIL',
        'branch': 'PF', 'part': PART_INDEX, 'status': 'STARTED',
        'shared_grid': SHARED_GRID,
        'pf_configuration': {'groups': PF_GROUPS, 'loops': PF_LOOPS, 'nfp': 1, 'native_mode': 'S', 'currents_A': PF_CURRENTS.tolist()},
        'companion_tf_configuration': {'physical_coils': 48, 'field_periods': 16, 'target_axis_field_T': 16.0, 'strict_edge_ripple_limit_percent': 1.0, 'expected_edge_ripple_percent': 0.7539178029174242},
        'tf_edge_ripple_gate': {'status': 'REFERENCE_ONLY_NOT_EVALUATED_ON_PF_MGRID', 'strict_pass_reference': True},
    }
    try:
        data_root, locate_errors = find_data_root(work)
        if data_root is None:
            report.update({'status': 'INPUT_BLOCKED', 'input_errors': locate_errors[-50:]})
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
            output = make_output_zip(work, work, report_path, None)
            print(f'PF_MGRID_PART{PART_INDEX}_STATUS=INPUT_BLOCKED')
            print(f'PF_MGRID_PART{PART_INDEX}_OUTPUT_ZIP={output}')
            raise WorkerFailure('INPUT_BLOCKED: no verified PF shared-grid data root found')
        valid, errors = validate_root(data_root)
        report.update({'data_root': str(data_root), 'input_manifest_pass': valid, 'input_errors': errors})
        if not valid:
            report['status'] = 'INPUT_BLOCKED'
            report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
            output = make_output_zip(work, data_root, report_path, None)
            print(f'PF_MGRID_PART{PART_INDEX}_INPUT_SHA256_FAIL')
            print(f'PF_MGRID_PART{PART_INDEX}_OUTPUT_ZIP={output}')
            raise WorkerFailure('INPUT_BLOCKED: PF manifest verification failed')
        for name in (COIL_FILE, INPUT_FILE, EXTCUR_FILE):
            if not (data_root / name).is_file():
                raise FileNotFoundError(data_root / name)
        report['coil_summary'] = parse_coil_summary(data_root / COIL_FILE)
        if report['coil_summary']['loops'] != PF_LOOPS or report['coil_summary']['groups'] != PF_GROUPS:
            raise WorkerFailure('INPUT_BLOCKED: PF chunk coil summary mismatch')
        xgrid = find_xgrid(search_roots(work), work)
        native_dir = None
        if xgrid is None:
            report.update({'status': 'NATIVE_XGRID_NOT_FOUND', 'native_xgrid': None, 'native_mgrid_present': False})
        else:
            print(f'PF_MGRID_PART{PART_INDEX}_XGRID_SOURCE={xgrid}', flush=True)
            native_dir = Path(tempfile.mkdtemp(prefix=f'pf-mgrid-part{PART_INDEX}-native-', dir=str(work)))
            for name in (COIL_FILE, INPUT_FILE, EXTCUR_FILE):
                shutil.copy2(data_root / name, native_dir / name)
            staged, environment = stage_xgrid(xgrid, native_dir)
            log_path = native_dir / f'xgrid_pf_part{PART_INDEX}.log'
            code, elapsed, timed_out = run_native(staged, native_dir / INPUT_FILE, native_dir, log_path, environment)
            result_path = native_dir / EXPECTED_MGRID
            report.update({'native_xgrid': xgrid, 'native_return_code': code, 'native_elapsed_s': elapsed, 'native_internal_timeout_enabled': env_seconds('PF_MGRID_XGRID_TIMEOUT_SECONDS', 0.0) > 0, 'native_mgrid_present': result_path.is_file(), 'native_log': str(log_path)})
            if timed_out:
                report['status'] = 'NATIVE_XGRID_TIMEOUT'
            elif code == 0 and result_path.is_file():
                report['native_mgrid'] = inspect_mgrid(result_path)
                report['native_pass'] = bool(report['native_mgrid']['valid'])
                report['status'] = 'PF_MGRID_PART_PASS' if report['native_pass'] else 'NATIVE_MGRID_INVALID'
            else:
                report['status'] = 'NATIVE_XGRID_FAILED'
        report['accepted_for_pf_assembly'] = bool(report.get('native_pass', False))
        report['accepted_for_total_field'] = False
        report['elapsed_s'] = time.time() - started
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
        output = make_output_zip(work, data_root, report_path, native_dir)
        print(f'PF_MGRID_PART{PART_INDEX}_GROUPS={",".join(PF_GROUPS)}')
        print(f'PF_MGRID_PART{PART_INDEX}_LOOPS={PF_LOOPS}')
        print('PF_MGRID_SHARED_GRID=R4-11_Z-3.5-3.5_KP36_IR201_JZ201_MODE_S')
        print('PF_MGRID_TF_CONFIGURATION=48_PHYSICAL_COILS_16_FIELD_PERIODS_16T_COMPANION')
        print(f'PF_MGRID_PART{PART_INDEX}_STATUS={report["status"]}')
        print(f'PF_MGRID_PART{PART_INDEX}_OUTPUT_ZIP={output}')
        if report['status'] != 'PF_MGRID_PART_PASS':
            if native_dir is not None:
                print_log_tail(native_dir / f'xgrid_pf_part{PART_INDEX}.log')
            raise WorkerFailure(f'PF MGRID Part {PART_INDEX} failed closed: {report["status"]}')
    except WorkerFailure:
        raise
    except Exception as exc:
        report.update({'status': 'RUNTIME_ERROR', 'error': repr(exc), 'elapsed_s': time.time() - started})
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
        output = make_output_zip(work, work, report_path, None)
        print(f'PF_MGRID_PART{PART_INDEX}_RUNTIME_ERROR={exc!r}')
        print(f'PF_MGRID_PART{PART_INDEX}_OUTPUT_ZIP={output}')
        raise


main()
